In [77]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.feature_selection import VarianceThreshold, RFE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import shap

In [78]:
data = pd.read_csv(r'D:\Creditcard_fraud_detection\data\processed\creditcard_featured_data.csv')
pd.set_option("display.max_columns",None)
pd.set_option("display.max_rows",100)
data.head()

,trans_date_trans_time,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,unix_time,merch_lat,merch_long,is_fraud,transaction_hour,transaction_day,transaction_dayofweek,transaction_month,transaction_year,hour_sin,hour_cos,dayofweek_sin,dayofweek_cos,is_weekend,is_night,is_business_hour,age,amt_log,amount_bucket,card_transaction_number,previous_transaction_amt,time_since_previous_transaction_minutes,card_merchant_count,is_new_merchant,card_city_count,is_new_city,card_category_count,is_new_category,customer_merchant_distance_km,previous_merch_lat,previous_merch_long,distance_from_previous_location_km,location_velocity_kmph,previous_avg_amount,previous_std_amount,amount_vs_historical_avg,amount_zscore
0,2019-01-01 00:04:08,"fraud_Stroman, Hudson and Erdman",gas_transport,94.63,Jennifer,Conner,F,4655 David Island,Dublin,PA,18917,40.3750,-75.2045,2158,Transport planner,1961-06-19,1325376248,40.653382,-76.152667,0,0,1,1,1,2019,0.0,1.0,0.781831,0.62349,0,1,0,57.535934,4.560487,medium,0,NaN,NaN,0,1,0,1,0,1,85.922643,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2019-01-01 00:05:08,fraud_Corwin-Collins,gas_transport,71.65,Steven,Williams,M,231 Flores Pass Suite 720,Edinburg,VA,22824,38.8432,-78.6003,6018,"Designer, multimedia",1947-08-21,1325376308,38.948089,-78.540296,0,0,1,1,1,2019,0.0,1.0,0.781831,0.62349,0,1,0,71.364819,4.285653,medium,0,NaN,NaN,0,1,0,1,0,1,12.766923,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2019-01-01 00:07:27,fraud_Kiehn Inc,grocery_pos,96.29,Jack,Hill,M,5916 Susan Bridge Apt. 939,Grenada,CA,96038,41.6125,-122.5258,589,Systems analyst,1945-12-21,1325376447,41.657520,-122.230347,0,0,1,1,1,2019,0.0,1.0,0.781831,0.62349,0,1,0,73.029432,4.577696,medium,0,NaN,NaN,0,1,0,1,0,1,25.059079,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2019-01-01 00:09:03,fraud_Beier-Hyatt,shopping_pos,7.77,Christopher,Castaneda,M,1632 Cohen Drive Suite 639,High Rolls Mountain Park,NM,88325,32.9396,-105.8189,899,Naval architect,1967-08-30,1325376543,32.863258,-106.520205,0,0,1,1,1,2019,0.0,1.0,0.781831,0.62349,0,1,0,51.340178,2.171337,very_low,0,NaN,NaN,0,1,0,1,0,1,66.021685,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2019-01-01 00:17:40,fraud_Pacocha-Bauch,shopping_pos,9.55,Susan,Washington,F,759 Erin Mount Suite 956,May,TX,76857,31.9571,-98.9656,1791,Corporate investment banker,1965-07-26,1325377060,31.626350,-98.610225,0,0,1,1,1,2019,0.0,1.0,0.781831,0.62349,0,1,0,53.434634,2.356126,very_low,0,NaN,NaN,0,1,0,1,0,1,49.806610,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [79]:
df = data.copy()

In [80]:
df.shape

(222873, 53)

In [81]:
df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])

df = df.sort_values("trans_date_trans_time").reset_index(drop=True)

print("Chronologically sorted:",df["trans_date_trans_time"].is_monotonic_increasing)

Chronologically sorted: True


In [82]:
df.columns

Index(['trans_date_trans_time', 'merchant', 'category', 'amt', 'first', 'last',
       'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop',
       'job', 'dob', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud',
       'transaction_hour', 'transaction_day', 'transaction_dayofweek',
       'transaction_month', 'transaction_year', 'hour_sin', 'hour_cos',
       'dayofweek_sin', 'dayofweek_cos', 'is_weekend', 'is_night',
       'is_business_hour', 'age', 'amt_log', 'amount_bucket',
       'card_transaction_number', 'previous_transaction_amt',
       'time_since_previous_transaction_minutes', 'card_merchant_count',
       'is_new_merchant', 'card_city_count', 'is_new_city',
       'card_category_count', 'is_new_category',
       'customer_merchant_distance_km', 'previous_merch_lat',
       'previous_merch_long', 'distance_from_previous_location_km',
       'location_velocity_kmph', 'previous_avg_amount', 'previous_std_amount',
       'amount_vs_historical_avg', 'amount_z

In [83]:
print(df["trans_date_trans_time"].dtype)
print(df["trans_date_trans_time"].min())
print(df["trans_date_trans_time"].max())

datetime64[us]
2019-01-01 00:04:08
2020-06-21 12:12:32


In [84]:
split_index = int(len(df) * 0.80)

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

In [85]:
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain period:")
print(train_df["trans_date_trans_time"].min())
print(train_df["trans_date_trans_time"].max())

print("\nTest period:")
print(test_df["trans_date_trans_time"].min())
print(test_df["trans_date_trans_time"].max())

Train shape: (178298, 53)
Test shape: (44575, 53)

Train period:
2019-01-01 00:04:08
2020-03-06 09:38:52

Test period:
2020-03-06 09:40:37
2020-06-21 12:12:32


In [86]:
X_train = train_df.drop(columns=["is_fraud"])
y_train = train_df["is_fraud"]

X_test = test_df.drop(columns=["is_fraud"])
y_test = test_df["is_fraud"]

In [87]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (178298, 52)
y_train: (178298,)
X_test: (44575, 52)
y_test: (44575,)


In [88]:
print("Training target distribution:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True))

Training target distribution:
is_fraud
0    172330
1      5968
Name: count, dtype: int64
is_fraud
0    0.966528
1    0.033472
Name: proportion, dtype: float64

Testing target distribution:
is_fraud
0    43037
1     1538
Name: count, dtype: int64
is_fraud
0    0.965496
1    0.034504
Name: proportion, dtype: float64


In [89]:
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_cols = X_train.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()

print("Categorical columns:")
print(categorical_cols)
print("\nNumerical columns:")
print(numerical_cols)

Categorical columns:
['merchant', 'category', 'first', 'last', 'gender', 'street', 'city', 'state', 'job', 'dob', 'amount_bucket']

Numerical columns:
['amt', 'zip', 'lat', 'long', 'city_pop', 'unix_time', 'merch_lat', 'merch_long', 'transaction_hour', 'transaction_day', 'transaction_dayofweek', 'transaction_month', 'transaction_year', 'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos', 'is_weekend', 'is_night', 'is_business_hour', 'age', 'amt_log', 'card_transaction_number', 'previous_transaction_amt', 'time_since_previous_transaction_minutes', 'card_merchant_count', 'is_new_merchant', 'card_city_count', 'is_new_city', 'card_category_count', 'is_new_category', 'customer_merchant_distance_km', 'previous_merch_lat', 'previous_merch_long', 'distance_from_previous_location_km', 'location_velocity_kmph', 'previous_avg_amount', 'previous_std_amount', 'amount_vs_historical_avg', 'amount_zscore']


In [90]:
corr_data = df[numerical_cols + ["is_fraud"]].copy()

corr_target = (
    corr_data.corr()["is_fraud"].drop("is_fraud").sort_values(key=np.abs, ascending=False)
)

print(corr_target)

amt                                        0.466953
previous_transaction_amt                   0.413096
amount_vs_historical_avg                   0.301925
amt_log                                    0.275390
previous_avg_amount                        0.257972
hour_cos                                   0.187873
is_business_hour                          -0.119541
time_since_previous_transaction_minutes   -0.101109
previous_std_amount                        0.096677
card_transaction_number                   -0.078142
card_city_count                           -0.078142
is_night                                   0.070451
card_category_count                       -0.064139
amount_zscore                              0.061200
location_velocity_kmph                     0.060291
is_new_category                            0.037126
transaction_hour                           0.031937
transaction_month                         -0.030343
age                                        0.029289
dayofweek_co

In [91]:
df['distance_from_previous_location_km'].value_counts()

distance_from_previous_location_km
102.528546    1
71.098792     1
96.167727     1
137.391803    1
70.369150     1
             ..
68.901459     1
32.349355     1
126.503878    1
107.402007    1
86.697822     1
Name: count, Length: 221890, dtype: int64

In [92]:
df['distance_from_previous_location_km'].isnull().sum()

np.int64(983)

RFE — Recursive Feature Elimination

In [93]:
X = df.select_dtypes(include="number").drop(columns=["is_fraud"],errors="ignore")
y = df["is_fraud"]


X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())

print("NaN:", X.isna().sum().sum())
print("Inf:", np.isinf(X).sum().sum())


X_rfe = X.sample(n=min(200_000, len(X)),random_state=42)
y_rfe = y.loc[X_rfe.index]

print("RFE data shape:", X_rfe.shape)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000,class_weight="balanced"))])

rfe = RFE(estimator=pipeline,n_features_to_select=30,importance_getter="named_steps.model.coef_")

rfe.fit(X_rfe, y_rfe)

rfe_features = X.columns[rfe.support_]

print("\nSelected features:")
for feature in rfe_features:
    print(feature)

NaN: 0
Inf: 0
RFE data shape: (200000, 40)

Selected features:
amt
zip
lat
long
unix_time
merch_lat
merch_long
transaction_hour
transaction_day
transaction_year
hour_sin
hour_cos
dayofweek_cos
is_night
is_business_hour
amt_log
card_transaction_number
previous_transaction_amt
time_since_previous_transaction_minutes
card_merchant_count
is_new_merchant
card_city_count
is_new_city
card_category_count
is_new_category
previous_merch_long
location_velocity_kmph
previous_avg_amount
previous_std_amount
amount_vs_historical_avg


In [94]:
constant_features = [
    col for col in X.columns
    if X[col].nunique(dropna=False) <= 1
]

print("Constant features:", constant_features)

X = X.drop(columns=constant_features)

Constant features: []


Mutual Information

use Mutual Information to find nonlinear relationships

In [95]:
X_numeric = X.select_dtypes(include="number").copy()
X_numeric = X_numeric.fillna(X_numeric.median())
mi = mutual_info_classif(X_numeric,y,random_state=42)
mi_scores = pd.Series(mi,index=X_numeric.columns).sort_values(ascending=False)
display(mi_scores.to_frame("MI_score"))

,MI_score
age,0.088602
amt,0.085619
amt_log,0.085571
previous_transaction_amt,0.068523
amount_vs_historical_avg,0.046951
is_new_merchant,0.044599
amount_zscore,0.042439
unix_time,0.038390
time_since_previous_transaction_minutes,0.037920
transaction_year,0.036319


In [96]:
# Remove highly redundant features

corr_matrix = X_numeric.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape),k=1).astype(bool))
high_corr = (upper.stack().sort_values(ascending=False))
display(high_corr.head(30))

card_transaction_number  card_city_count             1.000000
long                     merch_long                  0.999124
                         previous_merch_long         0.996756
merch_long               previous_merch_long         0.995887
lat                      merch_lat                   0.993587
                         previous_merch_lat          0.991312
merch_lat                previous_merch_lat          0.984925
zip                      long                        0.910024
                         merch_long                  0.909226
                         previous_merch_long         0.907143
card_merchant_count      is_new_merchant             0.904473
hour_cos                 is_business_hour            0.847335
card_city_count          card_category_count         0.845096
card_transaction_number  card_category_count         0.845096
transaction_dayofweek    is_weekend                  0.824946
amt                      amount_vs_historical_avg    0.812711
dayofwee

In [97]:
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)
train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()

test_df = df.iloc[val_end:].copy()

In [98]:
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (156011, 53)
Validation: (33431, 53)
Test: (33431, 53)


In [99]:
print("========== TRAIN ==========")

print(train_df["trans_date_trans_time"].min())

print(train_df["trans_date_trans_time"].max())


print("\n========== VALIDATION ==========")

print(val_df["trans_date_trans_time"].min())

print(val_df["trans_date_trans_time"].max())


print("\n========== TEST ==========")

print(test_df["trans_date_trans_time"].min())

print(test_df["trans_date_trans_time"].max())

========== TRAIN ==========
2019-01-01 00:04:08
2019-12-28 16:50:40

========== VALIDATION ==========
2019-12-28 16:52:36
2020-04-03 17:54:44

========== TEST ==========
2020-04-03 17:55:41
2020-06-21 12:12:32


In [100]:
for name, data in [("Train", train_df),("Validation", val_df),("Test", test_df)]:
    print(f"\n===== {name} =====")
    print(data["is_fraud"].value_counts())
    print(data["is_fraud"].value_counts(normalize=True).mul(100).round(4))


===== Train =====
is_fraud
0    150890
1      5121
Name: count, dtype: int64
is_fraud
0    96.7175
1     3.2825
Name: proportion, dtype: float64

===== Validation =====
is_fraud
0    32178
1     1253
Name: count, dtype: int64
is_fraud
0    96.252
1     3.748
Name: proportion, dtype: float64

===== Test =====
is_fraud
0    32299
1     1132
Name: count, dtype: int64
is_fraud
0    96.6139
1     3.3861
Name: proportion, dtype: float64


In [101]:
TARGET = "is_fraud"

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]
X_val = val_df.drop(columns=[TARGET])
y_val = val_df[TARGET]
X_test = test_df.drop(columns=[TARGET])
y_test = test_df[TARGET]

In [102]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (156011, 52)
y_train: (156011,)
X_val: (33431, 52)
y_val: (33431,)
X_test: (33431, 52)
y_test: (33431,)


In [103]:
for data in [ X_train,X_val,X_test]:
    data.drop(columns=["trans_date_trans_time"],
        errors="ignore",
        inplace=True)

In [104]:
train_final = X_train.copy()
train_final["is_fraud"] = y_train.values

validation_final = X_val.copy()
validation_final["is_fraud"] = y_val.values

test_final = X_test.copy()
test_final["is_fraud"] = y_test.values

In [105]:
train_final.to_csv("../data/processed/train_features.csv",index=False)

validation_final.to_csv("../data/processed/validation_features.csv",index=False)

test_final.to_csv("../data/processed/test_features.csv",index=False)

print("Train/validation/test datasets saved.")

Train/validation/test datasets saved.
